In [14]:
import time
import warnings
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display
import ipywidgets as widgets
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import lightkurve as lk
from lightkurve import search_lightcurve
import warnings
from astropy.units import UnitsWarning
import pprint


In [11]:
def fetchStitchedLC(KIC, quarters = 1):
    """
    Given a KIC, fetch its light curve data using Lightkurve.
    Returns: stitched LC object.
    """
    search = lk.search_lightcurve(
        f"KIC {KIC}",
        mission = "Kepler",
        author = "Kepler",
        cadence = "long"
    )    
    # Filter to get the last 'quarters' quarters of data
    if len(search) >= quarters:
        search = search[-quarters:]

    return search.download().remove_nans()


In [3]:
df = pd.read_csv(r"../assets/data/kepler-eclipsing-binary-catalog.csv")
df = df.drop(columns=["Unnamed: 11"])
# df = df.head(150)
df = df.set_index("KIC")


In [4]:
df


,period,period_err,bjd0,bjd0_err,morph,GLon,GLat,kmag,Teff,SC
KIC,,,,,,,,,,
3863594,0.053268,0.0,55000.000000,0.004327,0.79,-1.0000,-1.0000,-1.000,-1.0,False
10417986,0.073731,0.0,55000.027476,0.004231,0.99,81.0390,11.0820,9.128,-1.0,True
8912468,0.094838,0.0,54953.576945,0.005326,0.98,80.1095,7.8882,11.751,6194.0,False
8758716,0.107205,0.0,54953.672989,0.006197,1.00,77.7478,11.6565,13.531,-1.0,False
10855535,0.112782,0.0,54964.629315,0.006374,0.99,79.3949,15.9212,13.870,7555.0,False
...,...,...,...,...,...,...,...,...,...,...
9408440,989.985000,-1.0,55346.365980,0.096130,0.00,78.5607,12.2615,13.199,5688.0,False
8054233,1058.000000,-1.0,54751.806288,0.968052,0.03,78.6142,7.7321,11.783,4733.0,False
7672940,1064.270000,-1.0,54977.092960,0.089646,0.00,74.5296,14.6136,12.328,-1.0,False


In [6]:
# Step 1: Extract KIC values from the DataFrame
kic_values = df.index.tolist()


In [ ]:
kic_values


[3863594,
 10417986,
 8912468,
 8758716,
 10855535,
 9472174,
 9612468,
 6613627,
 5302006,
 9898401,
 7375612,
 5872696,
 7767774,
 12350008,
 10684673,
 9532219,
 6699679,
 6287172,
 11825204,
 4921906,
 6387887,
 8288741,
 12055255,
 11013201,
 8108785,
 1572353,
 10288502,
 10453521,
 9238207,
 5166136,
 8555795,
 6144827,
 10030943,
 10965091,
 2715417,
 11413213,
 9077796,
 6050116,
 12458797,
 6350020,
 7198474,
 7546791,
 7871200,
 7959612,
 3972629,
 9345163,
 8816790,
 12216817,
 11494583,
 4738426,
 8122124,
 9032671,
 5960283,
 9412114,
 9004380,
 4857282,
 12602985,
 11336707,
 12104285,
 8367007,
 11769739,
 4138301,
 9478836,
 12508348,
 5685072,
 3839964,
 12598713,
 9700154,
 9662581,
 9777987,
 2856960,
 5611561,
 7339345,
 9288175,
 9392331,
 9239684,
 12004834,
 11284547,
 8045121,
 6106771,
 10557008,
 5785551,
 9388303,
 5773205,
 2437038,
 7269843,
 6072578,
 8028158,
 2435971,
 9026766,
 10802917,
 3832382,
 7697065,
 9760531,
 5956588,
 3743834,
 9935311,
 1170

In [ ]:
# Step 2: Define the number of KICs to process (proof of concept)
num_kics_to_process = 5

lc_dict = {}

# Step 3: Iterate over a subset of each KIC value
for kic in tqdm(kic_values[:num_kics_to_process], desc = "Processing KICs"):
    # Fetch stitched light curve data for each KIC
    lc = fetchStitchedLC(kic, quarters = 2)
    
    # Extract time, flux, and flux_err arrays
    time = lc.time.value
    flux = lc.flux.value
    flux_err = lc.flux_err.value
    
    # Store the extracted data in the dictionary
    lc_dict[kic] = {"time": time, "flux": flux, "flux_err": flux_err}
    
    # lc_dict[kic] = lc
    
    # Perform further operations with the light curve data (e.g., analysis, plotting, etc.)
    # Example: Plotting the light curve
    # lc.plot()
    # display(lc)


Processing KICs:   0%|          | 0/5 [00:00<?, ?it/s]

/Users/wayfinder/Code/fault-in-our-stars/.env/lib/python3.13/site-packages/lightkurve/search.py:420: LightkurveWarning: Warning: 2 files available to download. Only the first file has been downloaded. Please use `download_all()` or specify additional criteria (e.g. quarter, campaign, or sector) to limit your search.
  warnings.warn(
/Users/wayfinder/Code/fault-in-our-stars/.env/lib/python3.13/site-packages/lightkurve/search.py:420: LightkurveWarning: Warning: 2 files available to download. Only the first file has been downloaded. Please use `download_all()` or specify additional criteria (e.g. quarter, campaign, or sector) to limit your search.
  warnings.warn(
/Users/wayfinder/Code/fault-in-our-stars/.env/lib/python3.13/site-packages/lightkurve/search.py:420: LightkurveWarning: Warning: 2 files available to download. Only the first file has been downloaded. Please use `download_all()` or specify additional criteria (e.g. quarter, campaign, or sector) to limit your search.
  warnings.w

In [16]:
pprint.pprint(lc_dict)


{3863594: <KeplerLightCurve length=3535 LABEL="KIC 3863594" QUARTER=16 AUTHOR=Kepler FLUX_ORIGIN=pdcsap_flux>
       time             flux      ...   pos_corr1      pos_corr2   
                    electron / s  ...      pix            pix      
       Time           float32     ...    float32        float32    
------------------ -------------- ... -------------- --------------
1472.1170971861065  1.0087238e+05 ...  5.6012170e-03 -2.1462505e-01
 1472.137530156877  1.0000698e+05 ...  5.5397763e-03 -2.1516806e-01
1472.1579629278858  1.0092012e+05 ...  5.5768704e-03 -2.1571192e-01
1472.1783958992455  1.0000207e+05 ...  5.7797516e-03 -2.1596017e-01
  1472.19882877083  1.0084972e+05 ...  5.4182606e-03 -2.1625915e-01
1472.2192615427703  1.0024647e+05 ...  5.5446364e-03 -2.1623993e-01
1472.2396945149449  1.0065079e+05 ...  5.4619168e-03 -2.1628410e-01
1472.2601273874607  1.0050985e+05 ...  5.4184557e-03 -2.1661688e-01
1472.2805601602158  1.0041942e+05 ...  5.4839822e-03 -2.1648978e-01
      

In [17]:
def plotLCFromDict(lc_dict, KIC):
    """
    Plot the light curve data for a given KIC from the dictionary.
    
    Parameters:
    - lc_dict: Dictionary containing KIC as keys and LC objects as values.
    - KIC: The KIC value for which to plot the light curve.
    """
    if KIC in lc_dict:
        lc = lc_dict[KIC]
        lc.plot()
        display(lc)
    else:
        print(f"No light curve data found for KIC {KIC}")


In [19]:
import random

# Sample a random KIC value from the list
random_kic = random.choice(kic_values)
print(f"Random KIC: {random_kic}")

# Plot the light curve for the sampled KIC
plotLCFromDict(lc_dict, random_kic)


Random KIC: 10551346
No light curve data found for KIC 10551346
